In [1]:
import numpy as np
import sys
sys.path.append("../../")
sys.path.append("../../Visualization/")
sys.path.append("../../../")

In [2]:
sys.path.append(".")

In [3]:
sys.path.append('../../periodic_patches/'); sys.path.append('../'); sys.path.append('../../../gmsh')
import inflation, sparse_matrices, mesh, numpy as np, importlib, pickle
import inflatables_parametrization as parametrization
from numpy.linalg import norm
from io_redirection import suppress_stdout
import visualization

In [4]:
import visualize_stiffness, importlib
importlib.reload(visualize_stiffness)

In [5]:
sys.path.append('periodic_patches/')
sys.path.append('gmsh')

In [6]:
import parametrization_experiment_helper, importlib
importlib.reload(parametrization_experiment_helper)

In [7]:
pattern = parametrization_experiment_helper.Pattern_data[2]

In [8]:
experiment_file = pattern['experiment_file']
stiffness_path = pattern['stiffness_path']
pattern_name = pattern['name']
num_pattern_params = pattern['num_pattern_params']
param_index = pattern['param_index']
default_param = pattern['default_param']
param_range = pattern['param_range']
param_normalization_factor = pattern['param_normalization_factor']
fusing_curve_polyline = pattern['fusing_curve_polyline_function']
                                

In [188]:
shape = parametrization_experiment_helper.Shape_data[0]

In [189]:
shape_name = shape['name']
shape_path = shape['path']

In [190]:
shape_name, pattern_name

### Overview

In [191]:
data_path = 'output/2024_01_18_00_03/{}_{}/'.format(shape_name, pattern_name)

In [192]:
import utils, mesh_utilities

In [193]:
# ### Parametrization
target_surf = mesh.Mesh(shape_path)
target_surf.setVertices(utils.prototypeScaleNormalization(target_surf.vertices(), placeAtopFloor=False))
target_surf = mesh_utilities.subdivide_loop(target_surf, 1)
target_surf.save(data_path + '/target_surf.obj')

In [194]:
import parametrization_helper

In [195]:
default_frequency = 0.2
default_mesh_size = 4
default_edge_soup_threshold = 1e0

scale = 1

frequency = default_frequency * scale
mesh_size = default_mesh_size / scale
edge_soup_threshold = default_edge_soup_threshold


In [196]:
sdfVertices, sdfTris, sdf, sheet_vxs, concatenated_polylines, sheet_edges_polylines,  boundaryVxs, boundaryEdges, upsampleMesh_vertices, upsampleMesh_triangles, upsampledAngles, upsampledPatternParams = parametrization_helper.get_polyline_from_pattern_parameters(_, fusing_curve_polyline, nsubdiv = 4, frequency=0.2, duplicates_removable_threshold=[1e-4, 1e-2, 1e-1, 1e0, 2e0], path = data_path, load_data = 'final_results')

In [197]:
# sdfVertices, sdfTris, sdf, sheet_vxs, concatenated_polylines, sheet_edges_polylines,  boundaryVxs, boundaryEdges, upsampleMesh_vertices, upsampleMesh_triangles, upsampledAngles, upsampledPatternParams = parametrization_helper.get_polyline_from_pattern_parameters(rparam, fusing_curve_polyline, nsubdiv = 4, frequency=default_frequency, duplicates_removable_threshold=[1e-4, 1e-2, 1e-1, 1e0, 2e0], path = data_path)

In [198]:
if len(boundaryEdges) > 1:
    concatenated_boundary_edges = []
    for polyline in boundaryEdges:
        concatenated_boundary_edges.extend(polyline)
    concatenated_boundary_edges = np.array(concatenated_boundary_edges)
    # Define a function to calculate the length of a sublist
    def sublist_length(sublist):
        return len(sublist)

    # Sort boundaryEdges in descending order of sublist length
    boundaryEdges = sorted(boundaryEdges, key=sublist_length, reverse=True)
else:
    concatenated_boundary_edges = np.array(boundaryEdges[0])


In [199]:
import matplotlib.pyplot as plt
import matplotlib as mpl

In [200]:
visualization.scalarFieldPlotFast(sdfVertices, sdfTris, sdf, width = 5, height=5)
visualization.plot_line_segments(sheet_vxs, concatenated_polylines, width = 5, height = 5)
visualization.plot_line_segments(list(sheet_vxs) + list(boundaryVxs), list(concatenated_polylines) + list(concatenated_boundary_edges + len(sheet_vxs)), width = 15, height = 15)
plt.scatter(boundaryVxs[concatenated_boundary_edges[:,0], 0], boundaryVxs[concatenated_boundary_edges[:,0], 1], c = np.arange(len(concatenated_boundary_edges[:, 0])), cmap = mpl.colormaps['Greys'])

## Meshing and inflation simulation

In [201]:
import mesher_helper
importlib.reload(mesher_helper)

In [202]:
import time
time_stamp = time.strftime("%Y_%m_%d_%H_%M")

In [203]:
importlib.reload(parametrization_helper)

In [204]:
from shapely import LineString

In [205]:
use_holes = True

In [206]:
if use_holes:
    boundary_holes_vxs, non_boundary_holes_vxs, boundary_polygons = parametrization_helper.post_process_holes(boundaryVxs, boundaryEdges, sheet_vxs, sheet_edges_polylines, smoothing = 5.0, area_threshold=4, avg_len = default_mesh_size / 2, distance_threshod=2)
else:
    selected_elements = [np.array(sublist)[:, 0] for sublist in boundaryEdges[1:]]
    holes_vxs = list(boundaryVxs[selected_elements])
    fused_vxs = []

In [207]:
boundary_polygons

In [208]:
if use_holes:
    plot_points = []
    plot_edges = []
    for polyline in boundary_holes_vxs:
        plot_edges.extend(np.array([[i, (i+1)%len(polyline)] for i in range(len(polyline))]) + len(plot_points))
        plot_points.extend(polyline)
        
    boundary_polyline = boundaryVxs[np.array(boundaryEdges[0])[:, 0]]
    
    plot_edges.extend(np.array([[i, (i+1)%len(boundary_polyline)] for i in range(len(boundary_polyline))]) + len(plot_points))
    plot_points.extend(boundary_polyline)
    
    visualization.plot_line_segments(plot_points, plot_edges, width = 10, height = 10)
    for polyline in non_boundary_holes_vxs:
        plot_edges.extend(np.array([[i, (i+1)%len(polyline)] for i in range(len(polyline))]) + len(plot_points))
        plot_points.extend(polyline)
    visualization.plot_line_segments(plot_points, plot_edges, width = 25, height = 25)

In [209]:
importlib.reload(mesher_helper)

In [210]:
boundary_curve = boundaryVxs[np.array(boundaryEdges[0])[:, 0]]
boundary_curve = parametrization_helper.smooth_polyline(boundary_curve, 0, default_mesh_size)[:-1]

In [211]:
if use_holes:
    v, f, fusing_data = mesher_helper.generate_mesh_non_periodic(default_mesh_size, boundary_curve, [], non_boundary_holes_vxs, [], [], gui = False)
else:
    v, f, fusing_data = mesher_helper.generate_mesh_non_periodic(default_mesh_size, boundary_curve, [], holes_vxs, sheet_vxs, concatenated_polylines, gui = False)


In [212]:
import MeshFEM

In [213]:
import numpy as np
import copy

# Use the function
new_v, new_f, new_fusing_without_boundary = parametrization_helper.remove_dangling_vertices(v, f - 1, fusing_data)
m = MeshFEM.mesh.Mesh(new_v, new_f)
new_fusing = copy.copy(new_fusing_without_boundary)
new_fusing[m.boundaryVertices()] = True

In [214]:
fusing_data, new_fusing

In [215]:
# m, iwv, iwbv = sheet_meshing.newMeshingAlgorithm(sdfVertices, sdfTris, sdf, SV, SE, triArea=1e0)


In [216]:
visualization.plot_2d_mesh(m, pointList=np.where(np.array(new_fusing) == 1)[0], width=10, height=10)


In [217]:
import inflation
isheet = inflation.InflatableSheet(m, new_fusing)

### Save pattern

In [218]:
import shapely
channelMargin = 11

In [219]:
V = m.vertices()
polylines = isheet.fusedRegionBooleanIntersectSheetBoundary()
shapely_boundaryEdges = shapely.MultiLineString([V[p] for p in polylines])
#utils.save(boundaryEdges.buffer(channelMargin), 'test.pkl.gz')
bypasses = shapely_boundaryEdges.buffer(channelMargin)
if bypasses.geom_type == 'Polygon': bypasses = [bypasses] # we generally expect a multipolygon...
outerAirChannelPolygons = [shapely.ops.unary_union([shapely.Polygon(boundaryVxs[np.array(boundaryEdges[0])[:, 0]])] + list(bypasses.geoms))]

In [220]:
smart_polygon = outerAirChannelPolygons[0]

In [221]:
coords = np.array(smart_polygon.exterior.coords)

In [222]:
len(coords)

In [226]:
plt.scatter(coords[:, 0], coords[:, 1])

In [227]:
polylines = []
for polyline in sheet_edges_polylines:
    polyline = np.array(polyline)
    polylines.append(sheet_vxs[np.array(list(polyline[:, 0]) + list([polyline[-1, 1]]))][:, :2].tolist())
parametrization_helper.save_to_obj(coords[:-1], polylines, '{}_{}_sheet_pattern_{}_margin_{}.obj'.format(shape_name, pattern_name, time_stamp, channelMargin))

### End

In [228]:
if not 'rparam' in globals():
    uv = np.load(data_path + '/rparam_uv.npy')
else:
    uv = rparam.uv()

In [229]:
from mesh_utilities import SurfaceSampler, tubeRemesh


paramSampler = SurfaceSampler(np.pad(uv, [(0, 0), (0, 1)], 'constant'), target_surf.triangles())
liftedSheetPositions = paramSampler.sample(m.vertices(), target_surf.vertices())

isheet.setUninflatedDeformation(liftedSheetPositions.transpose())

isheet.getVars()

In [230]:
import py_newton_optimizer
niter = 2000
iterations_per_output = 10
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.gradTol = 1e-10
opts.niter = iterations_per_output

In [231]:
# Are the flat region causing a problem? They might not actually control the metric...
# Try replacing them with single wall...
# Analyze the actual stretching factor (much easier to do with skeleton walls)

from tri_mesh_viewer import TriMeshViewer
from visualization import TriMeshViewerWithSurface
viewer = TriMeshViewerWithSurface(isheet, target_surf, width=768, height=640)
viewer.showWireframe(True)

viewer.show()

In [232]:
viewer.update(scalarField=utils.getStrains(isheet)[:, 0])    

In [233]:
import boundaries
bdryVars = boundaries.getOuterBoundaryVars(isheet)
fixedVars = bdryVars

In [234]:
isheet.setUseTensionFieldEnergy(True)

isheet.setUseHessianProjectedEnergy(False)

fixedVars, hessianShift = bdryVars, 1e-6

framerate = 20
def cb(it):
    if it % framerate == 0:
        viewer.update(scalarField=utils.getStrains(isheet)[:, 0])    

### First solve with low pressure to get out of indefinite state

In [235]:
isheet.pressure = 1e-6

In [236]:
opts.niter = 5

import time
cr = inflation.inflation_newton(isheet, fixedVars, opts, hessianShift = hessianShift, callback = cb)

### Then inflate

In [237]:
isheet.pressure = 0.025

In [238]:
import benchmark

In [239]:
opts.niter = 2000
opts.gradTol = 1e-7

import time
benchmark.reset()
cr = inflation.inflation_newton(isheet, fixedVars, opts, hessianShift = hessianShift, callback = cb)
benchmark.report()

isheet.tensionStateHistogram()

In [237]:
def export_top_bottom_mesh(isheet, export_path, shape_name, pattern_name):
    mesh_3d = isheet.visualizationMesh(True)
    mesh_2d = isheet.mesh()
    vx_3d = mesh_3d.vertices()
    elements_3d = mesh_3d.elements()

    new_mesh_3d = MeshFEM.Mesh(vx_3d[:mesh_2d.numVertices()], elements_3d[:mesh_2d.numElements()])

    new_mesh_3d.save(export_path + '{}_{}_mesh_3d_top.obj'.format(shape_name, pattern_name))

    new_mesh_3d = MeshFEM.Mesh(vx_3d[m.numVertices():], elements_3d[mesh_2d.numElements():] - mesh_2d.numVertices())
    new_mesh_3d.save(export_path + '{}_{}_mesh_3d_bottom.obj'.format(shape_name, pattern_name))

In [238]:
export_top_bottom_mesh(isheet, data_path, shape_name, pattern_name)

In [ ]:
# isheet.nu
# for i in range(len(isheet.mesh().vertices())):
#     if (isheet.varIdx(0, i) !=  isheet.varIdx(1, i)):
#         print(isheet.varIdx(0, i), isheet.varIdx(1, i))

In [ ]:
# Plot maximum tensile strains in the sheet to verify the pressure is reasonable
from matplotlib import pyplot as plt
plt.hist(utils.getStrains(isheet)[:, 0], bins=1000);
plt.xlim(-0.04, 0.1);

### Analyze curvature of the inflated surface

In [ ]:
isa = inflation.InflatedSurfaceAnalysis(isheet)
curvature = isa.curvature()
metric = isa.metric()

In [ ]:
import matplotlib, vis
from tri_mesh_viewer import TriMeshViewer
isurf = isa.inflatedSurface()
metric_vf = vis.fields.VectorField(isurf, metric.sigma_2[:, None] * metric.left_stretch, vmin=0, vmax=1.0,
                                   align=vis.fields.VectorAlignment.CENTER, colormap=matplotlib.cm.viridis,
                                   glyph=vis.fields.VectorGlyph.CYLINDER)

viewer2 = TriMeshViewer(isurf, width=768, height=640, scalarField=vis.fields.ScalarField(isurf, curvature.meanCurvature(), colormap=matplotlib.cm.coolwarm), vectorField=None)
# viewer2.showWireframe()
viewer2.show()